# 03 — Document Classification and Transfer Learning

**Question:** Can visual layout and texture distinguish invoices, letters, receipts, and forms?

This notebook first creates a deliberately small synthetic dataset so the entire workflow is visible. It then trains either a compact CNN or a pretrained ResNet-18. Synthetic accuracy is a learning result, not evidence of real-world performance.

## 1. Setup

Install `requirements-deep-learning.txt` before running. Fixing random seeds improves repeatability, although GPU operations can still have small nondeterministic differences.

In [1]:
from pathlib import Path
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageEnhance, ImageFont
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
HERE = Path.cwd()
if not (HERE / 'data').exists():
    HERE = HERE / '03_document_classifier'
DATA, OUTPUT = HERE / 'data', HERE / 'output'
GENERATED = DATA / 'generated'
OUTPUT.mkdir(exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

ModuleNotFoundError: No module named 'torch'

## 2. Inspect the downloaded real document image

A real dataset contains variation in scanners, templates, languages, aging, and page content. The downloaded page reminds us that the simple generated examples below are only a controlled starting point.

In [ ]:
real_page = Image.open(DATA / 'imageTextN.png').convert('RGB')
plt.figure(figsize=(13, 5)); plt.imshow(real_page); plt.title('Real sample page'); plt.axis('off');

## 3. Generate labeled documents

Each class gets characteristic words and slightly different layout. We vary rotation, contrast, and identifiers. The split is created before training: 16 train, 4 validation, and 4 test images per class. In real work, split by document source or template to prevent leakage.

In [ ]:
TEMPLATES = {
    'invoice': ['INVOICE', 'Invoice number:', 'Total due: EUR', 'Payment terms: 14 days'],
    'letter': ['Dear Dr. Miller,', 'Thank you for your letter.', 'Sincerely,', 'Ada Morgan'],
    'receipt': ['CORNER MARKET', 'Coffee      EUR 3.50', 'Bread       EUR 2.20', 'TOTAL       EUR 5.70'],
    'form': ['REGISTRATION FORM', 'Name:', 'Member ID:', '[ ] Standard  [ ] Premium'],
}

def get_font(size):
    candidates = ['/System/Library/Fonts/Supplemental/Arial.ttf',
                  '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf']
    for candidate in candidates:
        if Path(candidate).exists():
            return ImageFont.truetype(candidate, size)
    return ImageFont.load_default()

def render_document(kind, index, seed):
    rng = random.Random(seed)
    image = Image.new('RGB', (700, 500), 'white')
    draw = ImageDraw.Draw(image)
    draw.rectangle((28, 28, 672, 472), outline=(190, 190, 190), width=2)
    y = 65
    for line_number, base in enumerate(TEMPLATES[kind]):
        line = f'{base} {1000 + index}' if ':' in base else base
        draw.text((55 + rng.randint(-4, 4), y), line, fill='black',
                  font=get_font(34 if line_number == 0 else 25))
        y += 82 if line_number == 0 else 60
    image = image.rotate(rng.uniform(-3, 3), fillcolor='white')
    return ImageEnhance.Contrast(image).enhance(rng.uniform(0.75, 1.2))

split_sizes = {'train': 16, 'val': 4, 'test': 4}
for class_index, kind in enumerate(TEMPLATES):
    item_index = 0
    for split, count in split_sizes.items():
        destination = GENERATED / split / kind
        destination.mkdir(parents=True, exist_ok=True)
        for _ in range(count):
            render_document(kind, item_index, SEED + class_index * 1000 + item_index).save(
                destination / f'{kind}_{item_index:03d}.png')
            item_index += 1
print('Generated images:', sum(1 for _ in GENERATED.rglob('*.png')))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, kind in zip(axes, TEMPLATES):
    sample = next((GENERATED / 'train' / kind).glob('*.png'))
    axis.imshow(Image.open(sample)); axis.set_title(kind); axis.axis('off')
plt.tight_layout();

## 4. Transforms and data loaders

Training augmentation creates plausible variation each epoch. Validation and test transforms are deterministic. ImageNet normalization is required when using an ImageNet-pretrained backbone.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomRotation(3),
    transforms.ColorJitter(brightness=0.15, contrast=0.15), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

train_set = datasets.ImageFolder(GENERATED / 'train', train_transform)
val_set = datasets.ImageFolder(GENERATED / 'val', eval_transform)
test_set = datasets.ImageFolder(GENERATED / 'test', eval_transform)
loaders = {
    'train': DataLoader(train_set, batch_size=16, shuffle=True),
    'val': DataLoader(val_set, batch_size=16),
    'test': DataLoader(test_set, batch_size=16)}
print('classes:', train_set.classes, 'split sizes:', [len(train_set), len(val_set), len(test_set)])

## 5. Choose a model

The small CNN makes convolution/pooling visible and trains quickly. ResNet-18 demonstrates transfer learning. With a frozen backbone, only the new final layer receives gradient updates. Change `MODEL_CHOICE` and `FREEZE_BACKBONE` to run the comparison.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, class_count):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Linear(64, class_count)
    def forward(self, inputs):
        return self.classifier(self.features(inputs).flatten(1))

def make_model(choice, class_count, freeze_backbone=True):
    if choice == 'small_cnn':
        return SmallCNN(class_count)
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze_backbone:
        for parameter in model.parameters():
            parameter.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, class_count)
    return model

MODEL_CHOICE = 'small_cnn'  # change to 'resnet18'
FREEZE_BACKBONE = True
model = make_model(MODEL_CHOICE, len(train_set.classes), FREEZE_BACKBONE).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable parameters: {trainable:,} / {total:,}')

## 6. Training loop

Training enables gradients and parameter updates. Validation disables them and measures generalization. We retain the weights with the best validation accuracy rather than automatically keeping the final epoch.

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = correct = count = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            if training:
                loss.backward(); optimizer.step()
        total_loss += loss.item() * len(targets)
        correct += (outputs.argmax(1) == targets).sum().item()
        count += len(targets)
    return total_loss / count, correct / count

EPOCHS = 4
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-3)
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_state, best_val = copy.deepcopy(model.state_dict()), -1
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, loaders['train'], loss_fn, optimizer)
    val_loss, val_acc = run_epoch(model, loaders['val'], loss_fn)
    for key, value in [('train_loss', train_loss), ('val_loss', val_loss),
                       ('train_acc', train_acc), ('val_acc', val_acc)]: history[key].append(value)
    if val_acc > best_val:
        best_val, best_state = val_acc, copy.deepcopy(model.state_dict())
    print(f'epoch {epoch}: train loss={train_loss:.3f}, acc={train_acc:.3f}; '
          f'val loss={val_loss:.3f}, acc={val_acc:.3f}')
model.load_state_dict(best_state)

## 7. Evaluate once on the test split

Accuracy alone can hide a weak class. The classification report includes per-class precision, recall, and F1. The confusion matrix shows exactly which classes were confused.

In [ ]:
actual, predicted = [], []
model.eval()
with torch.no_grad():
    for inputs, targets in loaders['test']:
        predicted.extend(model(inputs.to(device)).argmax(1).cpu().tolist())
        actual.extend(targets.tolist())
print(classification_report(actual, predicted, target_names=test_set.classes, zero_division=0))
figure, axis = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(actual, predicted, display_labels=test_set.classes,
                                        cmap='Blues', ax=axis, colorbar=False)
figure.tight_layout(); figure.savefig(OUTPUT / 'confusion_matrix.png', dpi=150)
torch.save({'state_dict': model.state_dict(), 'classes': train_set.classes},
           OUTPUT / f'{MODEL_CHOICE}.pt')
print('Saved model and confusion matrix to', OUTPUT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['val_loss'], label='validation')
axes[0].set(title='Cross-entropy loss', xlabel='Epoch'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(history['train_acc'], label='train'); axes[1].plot(history['val_acc'], label='validation')
axes[1].set(title='Accuracy', xlabel='Epoch', ylim=(0, 1.05)); axes[1].legend(); axes[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUTPUT / 'learning_curves.png', dpi=150);

## Reflection

1. Is there a train/validation gap? What does it suggest?
2. Which class is hardest, and what evidence does the confusion matrix give?
3. Run frozen ResNet-18. Does pretrained visual knowledge help this tiny dataset?
4. Why would accuracy on these synthetic templates overestimate real performance?